In [4]:
import sys
sys.path.insert(0, "/home/ibm-user/shared/safecircle")
%load_ext autoreload
%autoreload 2

from src.risk import decide

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# (name, message score, transaction score, risky_category, EXPECTED action)
cases = [
    ("both high",                 0.9, 0.9, False, "hold_and_alert"),  # risk 0.90
    ("scam text + gift card",     0.9, 0.2, True,  "hold_and_alert"),  # risk 0.55, but the safety rule holds it
    ("odd payment, no scam text", 0.1, 0.8, False, "allow"),           # risk 0.45
    ("both low",                  0.1, 0.1, False, "allow"),           # risk 0.10
    ("right at the threshold",    0.6, 0.6, False, "hold_and_alert"),  # risk 0.60 counts as "at or above"
    # extras: prove the rule needs BOTH halves
    ("scam text, normal store",   0.9, 0.2, False, "allow"),
    ("key missing (Scorer stub)", 0.9, 0.2, None,  "allow"),
]

for name, m, t, risky, expected in cases:
    msg = {"score": m, "reasons": ["msg reason"]}
    txn = {"score": t, "reasons": ["txn reason"]}
    if risky is not None:
        txn["risky_category"] = risky
    d = decide(msg, txn)
    status = "PASS" if d["action"] == expected else "FAIL"
    print(f"{status}  {name:28} risk={d['risk']:.2f}  action={d['action']}")

PASS  both high                    risk=0.90  action=hold_and_alert
PASS  scam text + gift card        risk=0.60  action=hold_and_alert
PASS  odd payment, no scam text    risk=0.45  action=allow
PASS  both low                     risk=0.10  action=allow
PASS  right at the threshold       risk=0.60  action=hold_and_alert
PASS  scam text, normal store      risk=0.55  action=allow
PASS  key missing (Scorer stub)    risk=0.55  action=allow


In [6]:
import pandas as pd

REPO = "/home/ibm-user/shared/safecircle"
history = pd.read_csv(f"{REPO}/scenarios/history.csv")
history.columns = history.columns.str.strip().str.lower()

# "Usual" = the median, so one big past purchase doesn't skew it
usual_amount = history.groupby("user_id")["amount"].median()
usual_amount

user_id
linda     66.230
rosa      34.670
walter    47.125
Name: amount, dtype: float64

In [7]:
RISKY_CATEGORIES = {"gift_card", "wire", "money_transfer"}

def rule_score_from_features(amount_ratio, is_new_payee, risky_category):
    """The roadmap's rules: +0.4 new payee, +0.4 risky category, +0.3 if 5x usual amount, max 1.0."""
    score, reasons = 0.0, []
    if is_new_payee:
        score += 0.4
        reasons.append("new payee")
    if risky_category:
        score += 0.4
        reasons.append("gift-card or wire payment")
    if amount_ratio >= 5:
        score += 0.3
        reasons.append(f"{amount_ratio:.0f}x usual amount")
    return {"score": round(min(score, 1.0), 2), "reasons": reasons,
            "risky_category": bool(risky_category)}

def rule_score(txn):
    """Same input and output shape as score_transaction(), so it can stand in for it."""
    usual = usual_amount.get(txn["user_id"])
    ratio = float(txn["amount"]) / usual if usual else 0.0
    return rule_score_from_features(ratio,
                                    int(txn["is_new_payee"]) == 1,
                                    txn["merchant_category"] in RISKY_CATEGORIES)

In [8]:
from src.messages import score_message
from src.transactions import score_transaction

messages     = pd.read_csv(f"{REPO}/scenarios/messages.csv")
transactions = pd.read_csv(f"{REPO}/scenarios/transactions.csv")
messages.columns     = messages.columns.str.strip().str.lower()
transactions.columns = transactions.columns.str.strip().str.lower()
msg_text = messages.set_index("user_id")["text"]

TXN_KEYS = ["user_id", "txn_id", "timestamp", "amount", "merchant",
            "merchant_category", "is_new_payee"]

rows = []
for _, t in transactions.iterrows():
    txn   = t[TXN_KEYS].to_dict()
    msg   = score_message(msg_text.get(t["user_id"], ""))
    rules = rule_score(txn)
    model = score_transaction(txn)
    rows.append({
        "user":          t["user_id"],
        "rule_score":    rules["score"],
        "model_score":   round(float(model["score"]), 2),
        "action_rules":  decide(msg, rules)["action"],
        "action_model":  decide(msg, model)["action"],
        "expected":      t["expected_action"],
        "rule_reasons":  ", ".join(rules["reasons"]),
    })

comparison = pd.DataFrame(rows)
comparison

,user,rule_score,model_score,action_rules,action_model,expected,rule_reasons
0,rosa,1.0,1.00,hold_and_alert,hold_and_alert,hold_and_alert,"new payee, gift-card or wire payment, 43x usua..."
1,walter,0.0,0.00,allow,allow,allow,
2,linda,0.8,0.92,hold_and_alert,hold_and_alert,hold_and_alert,"new payee, gift-card or wire payment"


In [9]:
# (user, amount_ratio, is_new_payee, risky_category) as posted by the Scorer at Sync 2
sync2 = [
    ("rosa",   26.3, True,  True),
    ("walter",  0.9, False, False),
    ("linda",   3.3, True,  True),
]
for user, ratio, new, risky in sync2:
    print(user, rule_score_from_features(ratio, new, risky))

rosa {'score': 1.0, 'reasons': ['new payee', 'gift-card or wire payment', '26x usual amount'], 'risky_category': True}
walter {'score': 0.0, 'reasons': [], 'risky_category': False}
linda {'score': 0.8, 'reasons': ['new payee', 'gift-card or wire payment'], 'risky_category': True}
